# Creating metrics

#### Standard imports

In [ ]:
import numpy as np
import tables_io
import qp
import os
from astropy.stats import biweight_location, biweight_scale
from scipy.stats import sigmaclip

import matplotlib.pyplot as plt
from rail.plotting import pz_plotters, pz_dist_plotters

In [ ]:
DATADIR = '../public'
SUBMISSION_DIR = '../evaluation'

In [ ]:
TASKSETS = ["taskset_1", "taskset_2"]
SIMS = ["cardinal", "flagship"]
SCENARIOS = ["1yr", "10yr"]

In [ ]:
data_dict = {}
for taskset_ in TASKSETS:
    for sim_ in SIMS:
        for scenario_ in SCENARIOS:
            key = f"{taskset_}_{sim_}_{scenario_}"
            test_file = os.path.abspath(os.path.join(DATADIR, f"pz_challenge_{taskset_}_{sim_}_training_{scenario_}.hdf5"))
            validate_file = os.path.abspath(os.path.join(SUBMISSION_DIR, f"pz_challenge_{taskset_}_{sim_}_pz_evaluation_{scenario_}.hdf5"))
            data_dict[f"{key}_test"] = tables_io.read(test_file)
            data_dict[f"{key}_validate"] = qp.read(validate_file)            

#### Change this to be the dataset you want to validate

In [ ]:
to_validate = 'taskset_2_cardinal_10yr'
test_data, submit_data = data_dict[f'{to_validate}_test'], data_dict[f'{to_validate}_validate'] 

#### Point-estimate metrics

This makes three plots

1. A scatter plot of the point-estimate v. the reference redshift with metrics
2. A plot showing performance metrics as a function of redshfit
3. A plot showing performance metrics as a function of magnitude

In [ ]:
point_plotter = pz_plotters.PZPlotterPointEstimateVsTrueHist2D()
point_v_redshfit_plotter = pz_plotters.PZPlotterBiweightStatsVsRedshift()
point_v_mag_plotter = pz_plotters.PZPlotterBiweightStatsVsMag()

def point_metrics(prefix, test_data, submit_data):
    return point_plotter.run(        
        prefix,
        truth=test_data['redshift'],
        pointEstimate=np.squeeze(submit_data.ancil['zmode']),
        magnitude=test_data['mag_i_lsst'],
    )

def point_v_redshfit(prefix, test_data, submit_data):
    return point_v_redshfit_plotter.run(        
        prefix,
        truth=test_data['redshift'],
        pointEstimate=np.squeeze(submit_data.ancil['zmode']),
        magnitude=test_data['mag_i_lsst'],
    )

def point_v_mag(prefix, test_data, submit_data):
    return point_v_mag_plotter.run(        
        prefix,
        truth=test_data['redshift'],
        pointEstimate=np.squeeze(submit_data.ancil['zmode']),
        magnitude=test_data['mag_i_lsst'],
    )


In [ ]:
point_estimate_plots = point_metrics(to_validate, test_data, submit_data)

In [ ]:
point_v_redshift_plots = point_v_redshfit(to_validate, test_data, submit_data)

In [ ]:
point_v_mag_plots = point_v_mag(to_validate, test_data, submit_data)

#### Distribution metrics

This makes two plots

1. A plot showing the pdf of the $P(z_{ref} > z(q))$ 
3. A plot showing the cumulative distribuiton of $P(z_{ref} > z(q))$ 

In [ ]:
pit_prob_plotter = pz_dist_plotters.PZPlotterPITProb()

pit_qq_plotter = pz_dist_plotters.PZPlotterPITQQ()

def plot_pit_prob(prefix, test_data, submit_data):
    return pit_prob_plotter.run(
        prefix,
        truth=test_data['redshift'],
        pz=submit_data,
    )

def plot_pit_qq(prefix, test_data, submit_data):
    return pit_qq_plotter.run(
        prefix,
        truth=test_data['redshift'],
        pz=submit_data,
    )
    

In [ ]:
pit_prob_plots = plot_pit_prob(to_validate, test_data, submit_data)

In [ ]:
pit_qq_plots = plot_pit_qq(to_validate, test_data, submit_data)